In [11]:
import astropy.units as u
from astropy.coordinates import SkyCoord

In [12]:
!pip -q install reproject imageio

In [13]:
# 1. Instalación de librerías necesarias (si no las tenías instaladas)
import sys
!{sys.executable} -m pip install -q reproject imageio pandas numpy astropy

# 2. Importaciones necesarias
import os
import json
import logging
import numpy as np
import pandas as pd
import imageio.v2 as imageio
import astropy.units as u
from astropy.io import fits
from astropy.wcs import WCS
from astropy.time import Time
from astropy.coordinates import SkyCoord
from reproject import reproject_interp

# Silenciar errores menores de astropy
logging.getLogger('astropy').setLevel(logging.ERROR)

# 3. Definición de objetivos y función de movimiento propio
OBJETIVOS = {
    "r_dra":    dict(ra=248.1675,  dec=66.7550,   pmra=0,     pmdec=0,     epoca=2000.0, size=0.1),
    "rw_aur":   dict(ra=76.95642,  dec=30.40142,  pmra=0,     pmdec=0,     epoca=2000.0, size=0.1),
    "barnard":  dict(ra=269.45208, dec=4.69339,   pmra=-802,  pmdec=10362, epoca=2000.0, size=0.1),
    "wise0855": dict(ra=133.79513, dec=-7.24515,  pmra=-8120, pmdec=680,   epoca=2010.5, size=0.1),
    "eps_ind":  dict(ra=331.0438,  dec=-56.7827,  pmra=3960,  pmdec=-2540, epoca=2000.0, size=0.2),
}

def posicion_actual(clave, fecha="2026-01-01"):
    o = OBJETIVOS[clave]
    c = SkyCoord(ra=o["ra"] * u.deg, dec=o["dec"] * u.deg,
                 pm_ra_cosdec=o["pmra"] * u.mas / u.yr, 
                 pm_dec=o["pmdec"] * u.mas / u.yr,
                 obstime=Time(o["epoca"], format="jyear"), frame="icrs")
    hoy = c.apply_space_motion(new_obstime=Time(fecha))
    return SkyCoord(ra=hoy.ra, dec=hoy.dec, frame="icrs")

# 4. Rutas locales dentro de tu carpeta "Planet X"
BASE = "data/spherex_cache"    # Aquí debes tener tus carpetas con los FITS descargados
SALIDA = "../frontend/public/data"  # Aquí se guardarán los PNGs y JSONs generados

# 5. Función principal de procesamiento
def procesar(clave, detector="SPHEREx-D2", n=16, zoom=8):
    o = OBJETIVOS[clave]
    coord = posicion_actual(clave)
    size = o["size"]

    manifest_path = os.path.join(BASE, clave, "manifest.csv")
    if not os.path.exists(manifest_path):
        print(f"No se encontró el archivo manifest en: {manifest_path}")
        return

    m = pd.read_csv(manifest_path)
    m = m[m.detector == detector]
    m = m[[os.path.exists(os.path.join(BASE, clave, str(a))) for a in m.archivo]].sort_values("mjd")
    
    if len(m) == 0:
        print("Sin archivos para", clave, detector)
        return
        
    elegidos = np.unique(np.linspace(0, len(m) - 1, min(n, len(m))).astype(int))
    m = m.iloc[elegidos]

    escala = 6.15 / 3600
    npix = int(size / escala)
    destino = WCS(naxis=2)
    destino.wcs.ctype = ["RA---TAN", "DEC--TAN"]
    destino.wcs.crval = [coord.ra.deg, coord.dec.deg]
    destino.wcs.crpix = [npix / 2 + 0.5, npix / 2 + 0.5]
    destino.wcs.cdelt = [-escala, escala]

    d = detector.split("-")[-1]    # Ej: "D2"
    frames_dir = os.path.join(SALIDA, clave, "frames")
    os.makedirs(frames_dir, exist_ok=True)
    
    frames = []
    for i, fila in enumerate(m.itertuples()):
        archivo_path = os.path.join(BASE, clave, fila.archivo)
        try:
            with fits.open(archivo_path) as h:
                datos = h[1].data.astype(float)
                hdr = h[1].header
                w = WCS(hdr)
                try:                    # Longitud de onda en la posición del objetivo
                    x, y = w.world_to_pixel(coord)
                    sw = WCS(hdr, fobj=h, key="W")
                    sw.sip = None
                    lam, _ = sw.pixel_to_world(x, y)
                    lam = round(float(lam.to(u.micrometer).value), 4)
                except Exception:
                    lam = None
        except Exception as e:
            print("omitido", fila.archivo, e)
            continue

        img, _ = reproject_interp((datos, w), destino, shape_out=(npix, npix))
        img -= np.nanmedian(img)
        lo, hi = np.nanpercentile(img, [5, 99.5])
        img = np.nan_to_num(np.clip((img - lo) / (hi - lo), 0, 1)) * 255
        img = np.kron(np.flipud(img), np.ones((zoom, zoom))).astype(np.uint8)

        nombre = f"frames/{d}_{i:02d}.png"
        full_img_path = os.path.join(SALIDA, clave, nombre)
        os.makedirs(os.path.dirname(full_img_path), exist_ok=True)
        imageio.imwrite(full_img_path, img)
        
        frames.append(dict(
            url=nombre, 
            mjd=round(fila.mjd, 3),
            date=Time(fila.mjd, format="mjd").strftime("%Y-%m-%d"),
            detector=d, 
            wavelength_um=lam
        ))

    manifest = dict(
        target=dict(name=clave, ra=round(coord.ra.deg, 5), dec=round(coord.dec.deg, 5), fov=size),
        frames=frames
    )
    
    manifest_out_path = os.path.join(SALIDA, clave, f"manifest_{d}.json")
    os.makedirs(os.path.dirname(manifest_out_path), exist_ok=True)
    with open(manifest_out_path, "w") as f:
        json.dump(manifest, f, indent=1)
        
    print(f"¡Éxito! {clave} ({d}): {len(frames)} cuadros generados en {SALIDA}/{clave}")

# 6. Ejecutar la función con tu objetivo de prueba
procesar("r_dra")

"c:\Users\TiagoTZ\Desktop\Proyectos\Planet" no se reconoce como un comando interno o externo,
programa o archivo por lotes ejecutable.


¡Éxito! r_dra (D2): 16 cuadros generados en ../frontend/public/data/r_dra
